In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import DoubleType, FloatType

silver = spark.table("workspace.silver.gho_clean")
required = ["country_code", "region", "year", "indicator_code", "numeric_value"]

In [0]:
FLOAT_TYPES = (DoubleType, FloatType)


def _is_missing(column: str, dtype):
    col = F.col(column)
    if isinstance(dtype, FLOAT_TYPES):
        return col.isNull() | F.isnan(col)
    return col.isNull()


def count_missing(df: DataFrame, columns: list[str]) -> dict[str, int]:
    schema = df.schema
    row = df.agg(
        *[F.sum(_is_missing(c, schema[c].dataType).cast("int")).alias(c) for c in columns]
    ).first()
    return {c: row[c] or 0 for c in columns}


def assert_no_nulls(df: DataFrame, columns: list[str], table_name: str) -> None:
    failures = {c: n for c, n in count_missing(df, columns).items() if n > 0}
    assert not failures, f"{table_name}: nulls in required columns: {failures}"

In [0]:
assert silver.count() > 0, "silver is empty"

In [0]:
assert_no_nulls(silver, required, "silver")

string_cols = [c for c, t in silver.dtypes if t == "string" and c in required]
row = silver.agg(
    *[F.sum((F.trim(F.col(c)) == "").cast("int")).alias(c) for c in string_cols]
).first()
empty = {c: n for c, n in row.asDict().items() if n}
assert not empty, f"silver: empty strings in required columns: {empty}"

In [0]:
n_duplicates = (silver.groupBy("country_code", "year", "indicator_code")
    .count().filter("count > 1").count())
assert n_duplicates == 0, f"{n_duplicates} duplicated keys"

In [0]:
n_indicators = silver.select("indicator_code").distinct().count()
assert n_indicators == 15, f"{n_indicators} indicators, expected 15"

In [0]:
dtype = dict(silver.dtypes)["numeric_value"]
assert dtype == "double", f"numeric_value is {dtype}, expected double"